# Process the .xls file for cities with >100k population

Process the file table08.xls from United Nation's Department of Economic and Social Affairs [Statistics Division](https://unstats.un.org/unsd/demographic-social/products/dyb/dyb_2022/).

## Setup

In [32]:
%run -i "functions.py"

## Process the UN table

In [33]:
un_list = pd.read_excel(
    "../external_data/table08.xls",
    sheet_name = "Data",
    skiprows = 6,
    usecols = "A:B,J",
    engine = "xlrd"
).rename(columns = {"Unnamed: 0": "name", "Unnamed: 1": "pop_city", "Unnamed: 9": "pop_urban_area"})
un_list

,name,pop_city,pop_urban_area
0,AFRICA - AFRIQUE,NaN,NaN
1,Algeria - Algérie,NaN,NaN
2,16 IV 2008 (CDJC),NaN,NaN
3,Adrar,200834,...
4,Ain Defla,450280,...
...,...,...,...
4439,16 XI 2020 (CDFC),NaN,NaN
4440,PORT VILA,49034,...
4441,Wallis and Futuna Islands - Îles Wallis et Futuna,NaN,NaN
4442,21 VII 2008 (CDFC),NaN,NaN


In [35]:
records = []

current_continent = ""
current_country = ""
city = ""

info_row = re.compile(r"^\d") # Rows stating the source and year start with a digit
footnote = re.compile(r"\d+$") # Footnotes are digits at the end of the city name

cities_with_no_population_data = [] # List to keep track of cities with no population data


for idx, row in tqdm(un_list.iterrows(), total=len(un_list), position=0, leave=True):

    if pd.isna(row["name"]):
        continue


    caps = row["name"].isupper()
    

    # Check if it is a continent (all caps with no population data and the first element is not a digit):
    if caps and pd.isna(row["pop_city"]) and info_row.match(row["name"]) is None:
        continent = re.sub(r"[^a-zA-Z ]", "", row["name"].split(" - ")[0])
        current_continent = continent.title()
        print(f"Processing continent: {continent}")
        continue


    # Check if it is a country row (no population data and the first element is not a digit):
    if info_row.match(row["name"]) is None and pd.isna(row["pop_city"]):
        country = footnote.sub("", row["name"].split(" - ")[0]).strip()
        current_country = country.title()
        continue


    # Check if it is an information row (the first element is a digit):
    if info_row.match(row["name"]) is not None:
        continue

    capital = True if caps else False

    city = re.sub(r"\d+$", "", re.sub(r"\(.*?\)", "", row["name"])).strip().title() # Clean the city name: remove footnotes, parenthetical information, and write with uppercase first letter of each word

    if pd.isna(row["pop_city"]) or not isinstance(row["pop_city"], (int, float)):
        if pd.isna(row["pop_urban_area"]) or not isinstance(row["pop_urban_area"], (int, float)):
            cities_with_no_population_data.append(city)
            continue
        else:
            pop = int(row["pop_urban_area"])
    else:
        pop = int(row["pop_city"])
    

    records.append({
        "continent": current_continent,
        "country": current_country,
        "city": city,
        "is_capital": capital,
        "population": pop
    })

100%|██████████| 4444/4444 [00:00<00:00, 49559.00it/s]

Processing continent: AFRICA
Processing continent: AMERICA NORTH
Processing continent: AMERICA SOUTH
Processing continent: ASIA
Processing continent: EUROPE
Processing continent: OCEANIA


In [36]:
un_list_clean = pd.DataFrame(
    records,
    columns = ["continent", "country", "city", "is_capital", "population"]
)
un_list_clean

,continent,country,city,is_capital,population
0,Africa,Algeria,Adrar,False,200834
1,Africa,Algeria,Ain Defla,False,450280
2,Africa,Algeria,Ain Temouchent,False,299341
3,Africa,Algeria,Algiers,True,2712944
4,Africa,Algeria,Annaba,False,442230
...,...,...,...,...,...
4020,Oceania,Solomon Islands,Honiara,True,64609
4021,Oceania,Tonga,Nuku'Alofa,True,34142
4022,Oceania,Tuvalu,Funafuti,True,6320
4023,Oceania,Vanuatu,Port Vila,True,49034


In [37]:
# Perform some character correction
un_list_clean['city'] = (
    un_list_clean['city']
    .str.replace("ş", "ș", regex=False)
    .str.replace("Ş", "Ș", regex=False)
    .str.replace("ţ", "ț", regex=False)
    .str.replace("Ţ", "Ț", regex=False)
)

# Filter for cities with more than 100,000 population (the UN list includes cities where either the city of the urban area has more than 100,000 population)
un_list_clean = un_list_clean[un_list_clean["population"] >= 100000].reset_index(drop=True)
print(un_list_clean.shape[0])

3948


In [38]:
# Save the list
un_list_clean.to_csv("../external_data/table08_clean.csv", index=False)

## EUROPE

Our definition of Europe includes (39):
* Albania
* Andorra
* Austria
* Belarus
* Belgium
* Bosnia and Herzegovina
* Bulgaria
* Croatia
* Czech Republic
* Denmark
* Estonia
* Finland
* France
* Germany
* Greece
* Hungary
* Iceland
* Ireland
* Italy
* Kosovo
* Latvia
* Lithuania
* Luxembourg
* Moldova
* Montenegro
* Netherlands
* North Macedonia
* Norway
* Poland
* Portugal
* Romania
* Serbia
* Slovakia
* Slovenia
* Spain
* Sweden
* Switzerland
* Ukraine
* United Kingdom

and excludes (11):
* Armenia
* Azerbaijan
* Cyprus
* Georgia
* Liechtenstein
* Malta
* Monaco
* Russia
* San Marino
* Turkiye
* Vatican City

### Import data

In [147]:
# Import the list of European capitals already extracted
european_capitals = pd.read_csv("../cities/european_capitals.csv", sep = ";")
capitals = european_capitals['name_en'].tolist()
countries = sorted(european_capitals['country_en'].tolist())

In [148]:
# Import the cleaned UN list and filter for European countries
europe = pd.read_csv("../external_data/table08_clean.csv", sep = ",").query("continent == 'Europe'").drop(columns="continent").reset_index(drop=True)
europe

,country,city,is_capital,population
0,Albania,Durrës,False,113249
1,Albania,Tirana,True,418495
2,Austria,Graz,False,288806
3,Austria,Innsbruck,False,132110
4,Austria,Klagenfurt,False,100817
...,...,...,...,...
659,United Kingdom Of Great Britain And Northern I...,Reading,False,218705
660,United Kingdom Of Great Britain And Northern I...,Sheffield,False,518090
661,United Kingdom Of Great Britain And Northern I...,Southampton,False,253651
662,United Kingdom Of Great Britain And Northern I...,Stoke-On-Trent,False,270726


In [149]:
europe['city'].to_csv("../external_data/table08_europe_cities.csv", index=False, header=False)

We prompted Claude to check each individal city name for the English spelling. When there was no English spelling, we kept the original name.

In [150]:
cities_eng = pd.read_csv("../external_data/table08_europe_cities_english.csv").drop_duplicates()
europe = europe.merge(
    cities_eng,
    on="city",
    how="left"
)
europe

,country,city,is_capital,population,city_eng
0,Albania,Durrës,False,113249,Durrës
1,Albania,Tirana,True,418495,Tirana
2,Austria,Graz,False,288806,Graz
3,Austria,Innsbruck,False,132110,Innsbruck
4,Austria,Klagenfurt,False,100817,Klagenfurt
...,...,...,...,...,...
659,United Kingdom Of Great Britain And Northern I...,Reading,False,218705,Reading
660,United Kingdom Of Great Britain And Northern I...,Sheffield,False,518090,Sheffield
661,United Kingdom Of Great Britain And Northern I...,Southampton,False,253651,Southampton
662,United Kingdom Of Great Britain And Northern I...,Stoke-On-Trent,False,270726,Stoke-On-Trent


In [151]:
# Manual change to account for Claude inconsistencies
europe.query("city != city_eng")

,country,city,is_capital,population,city_eng
7,Austria,Wien,True,1897491,Vienna
23,Belgium,Antwerpen,False,498473,Antwerp
24,Belgium,Brugge,False,117260,Bruges
25,Belgium,Bruxelles,True,174383,Brussels
27,Belgium,Gent,False,248358,Ghent
45,Czechia,Plzen,False,174007,Pilsen
46,Czechia,Praha,True,1301432,Prague
47,Denmark,Ålborg,False,219476,Aalborg
48,Denmark,Århus,False,352315,Aarhus
51,Denmark,Kobenhavn,True,638790,Copenhagen


### Match the different sources

In [152]:
# Rename the countries to match the european_capital.csv
europe['country'] = europe['country'].replace({
    'United Kingdom Of Great Britain And Northern Ireland': 'United Kingdom',
    'Netherlands (Kingdom Of The)': 'Netherlands',
    "Republic Of Moldova": "Moldova",
    "Czechia": "Czech Republic"
})

In [153]:
# Checks
print("Countries in the UN list:")
print(set(europe['country'].unique()))

print()
print("Countries in the European capitals list that are not in the UN list:")
print(set(countries) - set(europe['country'].unique()))

print()
print("Countries in the UN list that are not in the European capitals list:")
print(set(europe['country'].unique()) - set(countries))

Countries in the UN list:
{'Romania', 'Estonia', 'Albania', 'Slovenia', 'Luxembourg', 'Czech Republic', 'Iceland', 'Norway', 'Netherlands', 'Switzerland', 'Bulgaria', 'Hungary', 'Belgium', 'France', 'Russian Federation', 'Serbia', 'Italy', 'Belarus', 'Slovakia', 'Spain', 'Lithuania', 'Ireland', 'Finland', 'North Macedonia', 'Latvia', 'Greece', 'Austria', 'Croatia', 'Portugal', 'Montenegro', 'Poland', 'Denmark', 'Moldova', 'Germany', 'United Kingdom', 'Ukraine', 'Sweden'}

Countries in the European capitals list that are not in the UN list:
{'Bosnia and Herzegovina', 'Andorra', 'Kosovo'}

Countries in the UN list that are not in the European capitals list:
{'Russian Federation'}


In [154]:
# Filter the UN list for the countries in the European capitals list
europe_filtered = europe.query("country in @countries").reset_index(drop=True)
europe_filtered

,country,city,is_capital,population,city_eng
0,Albania,Durrës,False,113249,Durrës
1,Albania,Tirana,True,418495,Tirana
2,Austria,Graz,False,288806,Graz
3,Austria,Innsbruck,False,132110,Innsbruck
4,Austria,Klagenfurt,False,100817,Klagenfurt
...,...,...,...,...,...
486,United Kingdom,Reading,False,218705,Reading
487,United Kingdom,Sheffield,False,518090,Sheffield
488,United Kingdom,Southampton,False,253651,Southampton
489,United Kingdom,Stoke-On-Trent,False,270726,Stoke-On-Trent


In [155]:
# Checks
print("Countries in the European capitals list that are not in the filtered:")
print(set(countries) - set(europe_filtered['country'].unique()))

print()
print("Countries in the filtered list that are not in the European capitals list:")
print(set(europe_filtered['country'].unique()) - set(countries))

Countries in the European capitals list that are not in the filtered:
{'Bosnia and Herzegovina', 'Andorra', 'Kosovo'}

Countries in the filtered list that are not in the European capitals list:
set()


In [156]:
# Rename the capitals to match the european_capital.csv
europe_filtered['city_eng'] = europe_filtered['city_eng'].replace({
    "Andorra La Vella": "Andorra la Vella",
    "Reykjavik": "Reykjavík",
    "Luxembourg City": "Luxembourg",
    'Chisinau': "Chișinău"
})

In [157]:
europe_capitals = europe_filtered.query("is_capital == True").reset_index(drop=True)
print("Capitals in the European capitals list that are not in the filtered:")
print(set(capitals) - set(europe_capitals['city_eng'].unique()))

print()
print("Capitals in the filtered list that are not in the European capitals list:")
print(set(europe_capitals['city_eng'].unique()) - set(capitals))

Capitals in the European capitals list that are not in the filtered:
{'Sarajevo', 'Pristina', 'Andorra la Vella'}

Capitals in the filtered list that are not in the European capitals list:
set()


In [160]:
# Remove capitals
europe_non_capitals = europe_filtered.query("is_capital == False").reset_index(drop=True)
europe_non_capitals

,country,city,is_capital,population,city_eng
0,Albania,Durrës,False,113249,Durrës
1,Austria,Graz,False,288806,Graz
2,Austria,Innsbruck,False,132110,Innsbruck
3,Austria,Klagenfurt,False,100817,Klagenfurt
4,Austria,Linz,False,205726,Linz
...,...,...,...,...,...
450,United Kingdom,Reading,False,218705,Reading
451,United Kingdom,Sheffield,False,518090,Sheffield
452,United Kingdom,Southampton,False,253651,Southampton
453,United Kingdom,Stoke-On-Trent,False,270726,Stoke-On-Trent


### Manual add the missing countries

There a few european countries missing from the UN list:
- Bosnia and Herzegovina
- Kosovo
- Andorra

We add the cities with more than 100k citizens from [Wikipedia](https://en.wikipedia.org/wiki/List_of_towns_and_cities_with_100,000_or_more_inhabitants/country:_A-B).

In this case, there is only one, since the other cities from these three countries with more than 100k citizens are the capitals. 

In [161]:
europe_non_capitals.loc[-1] = ["Bosnia And Herzegovina", "Banja Luka", True, 135059, "Banja Luka"]
europe_non_capitals.index = europe_non_capitals.index + 1  # shifting index
europe_non_capitals = europe_non_capitals.sort_values("city_eng").reset_index(drop=True)
europe_non_capitals

,country,city,is_capital,population,city_eng
0,Spain,A Coruña,False,245541,A Coruña
1,Germany,Aachen,False,247380,Aachen
2,Denmark,Ålborg,False,219476,Aalborg
3,Denmark,Århus,False,352315,Aarhus
4,United Kingdom,Aberdeen,False,207932,Aberdeen
...,...,...,...,...,...
451,Ukraine,Zhytomyr,False,260367,Zhytomyr
452,Poland,Zielona Góra,False,141631,Zielona Góra
453,Netherlands,Zoetermeer,False,124025,Zoetermeer
454,Switzerland,Zürich,False,423193,Zurich


### Extract final csv

In [163]:
european_100000pop = europe_non_capitals.drop(
        columns = ["is_capital"]
    ).rename(
        columns = {
            "country": "country_en",
            "city_eng": "name_en",
            "city": "name"
        }
    )

european_100000pop['nominatim_query'] = european_100000pop['name_en']
european_100000pop["alpha-2"] = european_100000pop["country_en"].map(get_alpha2)
european_100000pop

,country_en,name,population,name_en,nominatim_query,alpha-2
0,Spain,A Coruña,245541,A Coruña,A Coruña,ES
1,Germany,Aachen,247380,Aachen,Aachen,DE
2,Denmark,Ålborg,219476,Aalborg,Aalborg,DK
3,Denmark,Århus,352315,Aarhus,Aarhus,DK
4,United Kingdom,Aberdeen,207932,Aberdeen,Aberdeen,GB
...,...,...,...,...,...,...
451,Ukraine,Zhytomyr,260367,Zhytomyr,Zhytomyr,UA
452,Poland,Zielona Góra,141631,Zielona Góra,Zielona Góra,PL
453,Netherlands,Zoetermeer,124025,Zoetermeer,Zoetermeer,NL
454,Switzerland,Zürich,423193,Zurich,Zurich,CH


In [ ]:
european_100000pop = european_100000pop[["name_en", "country_en", "nominatim_query", "name", "alpha-2", "population"]]
european_100000pop.to_csv("../cities/european_100000pop.csv", index=False)

In [ ]:
european_100000pop_comments = copy.deepcopy(european_100000pop)
european_100000pop_comments['comments'] = [""]*(len(european_100000pop_comments))
european_100000pop_comments.to_excel("../cities/european_100000pop_comments.xlsx", index=False)